# Spectral DNA Fingerprinting via Sparse Autoencoders
### Commodity Alpha Generation from Hyperspectral Signatures

**Run this in Colab: Runtime -> Change runtime type -> choose GPU (T4) or leave as CPU.**
This SAE is small (<1M params); it trains in under a minute even on Colab's free CPU. GPU only helps once you scale to a real multi-million-pixel Pixxel cube with latent_dim in the 10,000+ range.

**What this notebook does:**
1. Loads/simulates physically-informed hyperspectral pixel spectra (swap in a real Pixxel L2A / AVIRIS cube by replacing one function — see `load_real_envi_cube` stub)
2. Trains a Sparse Autoencoder (L1 and TopK variants, both from real published interpretability research — Bricken et al. 2023 / Gao et al. 2024 — applied here to a new domain)
3. Builds an interpretable feature dashboard: maps unsupervised latent features to known USGS-style absorption lines
4. Compares raw-band vs PCA vs SAE-code downstream classification (including a subpixel-mixing/superposition stress test)
5. Feeds the interpretable stress signal into a commodity finance layer: yield-anomaly forecast -> supply-shock price impact -> Black-Scholes Greeks re-pricing -> Jensen's Alpha portfolio backtest

**Honesty note:** the SAE math (encoder/decoder architecture, L1/TopK sparsity) is standard, published, and well-validated — that's why it's trustworthy to build on. What's novel here is applying it to hyperspectral remote-sensing spectra instead of LLM activations, and wiring the resulting interpretable features into a trading pipeline. Not an unpublished algorithm — a genuinely underexplored application of one.

In [ ]:
!pip install -q torch scikit-learn scipy matplotlib
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

## 1. Physically-informed spectral simulator
(paste of `spectral_physics_simulator.py` — or `!wget`/upload it and `import` if you keep the file separate)

In [ ]:
# === PASTE spectral_physics_simulator.py CONTENTS HERE (or upload the file to Colab and `from spectral_physics_simulator import *`) ===
# For brevity in this notebook cell, upload src/spectral_physics_simulator.py via the Colab file browser,
# then run:
# from spectral_physics_simulator import simulate_dataset, WAVELENGTHS, USGS_LIKE_LIBRARY
import sys
sys.path.append('/content')  # if you uploaded the src/ files here
from spectral_physics_simulator import simulate_dataset, WAVELENGTHS, USGS_LIKE_LIBRARY

X, y, sev, wl = simulate_dataset(n_per_class=800, seed=42)
print('Dataset:', X.shape, 'classes:', sorted(set(y)))

## 2. Train the real PyTorch Sparse Autoencoder

In [ ]:
from spectral_sae import L1SparseAutoencoder, TopKSparseAutoencoder, train_sae, encode
import numpy as np

n = X.shape[0]
perm = np.random.permutation(n)
split = int(0.85 * n)
Xtr, Xval = X[perm[:split]].astype(np.float32), X[perm[split:]].astype(np.float32)

# --- Option A: L1-SAE (classic, tune l1_coeff) ---
sae = L1SparseAutoencoder(n_bands=X.shape[1], latent_dim=2048, l1_coeff=3e-3)
sae, history = train_sae(sae, Xtr, Xval, epochs=100, batch_size=512, lr=1e-3)

# --- Option B: TopK-SAE (no L1 tuning needed, often cleaner features) ---
# sae = TopKSparseAutoencoder(n_bands=X.shape[1], latent_dim=2048, k=32)
# sae, history = train_sae(sae, Xtr, Xval, epochs=100, batch_size=512, lr=1e-3)

Z = encode(sae, X.astype(np.float32))
print('Latent codes:', Z.shape, '| mean active features per pixel:', (Z > 0).sum(1).mean())

## 3. Interpretability: map latent features to known absorption lines (feature dashboard)

In [ ]:
from feature_interpretation import build_feature_dashboard, print_dashboard
dashboard = build_feature_dashboard(Z, X, wl, labels=y, top_n=30, min_active_pixels=15)
print(f'{len(dashboard)} interpretable monosemantic spectral features (MSFs) discovered, unsupervised')
print_dashboard(dashboard, k=25)

## 4. Downstream classification: raw bands vs PCA vs SAE codes (incl. subpixel superposition test — see `downstream_classifier.py` and the mixing-fraction regression cell in the README)

In [ ]:
from downstream_classifier import run_comparison, print_comparison
results = run_comparison(X, Z, y, sev, n_pca=32)
print_comparison(results)

## 5. Finance layer: stress score -> yield anomaly -> supply shock -> Greeks -> Jensen's Alpha

In [ ]:
from commodity_alpha_pipeline import (stress_score_to_yield_anomaly, yield_anomaly_to_price_impact,
    greeks_scenario_table, simulate_commodity_portfolio, build_signal_series_from_stress)

# Aggregate a region's disease-matched MSF activations into one stress score in [0,1]
disease_msf_ids = [d['feature_id'] for d in dashboard if d['dominant_class'] == 'diseased_crop']
regional_stress_score = float(np.clip(Z[:, disease_msf_ids].mean() * 5, 0, 1)) if disease_msf_ids else 0.2
print('Regional stress score:', regional_stress_score)

yield_anom = stress_score_to_yield_anomaly(regional_stress_score)
price_impact = yield_anomaly_to_price_impact(yield_anom)
print(f'Forecast yield anomaly: {yield_anom*100:.1f}% | forecast price impact: {price_impact*100:+.1f}%')

rows = greeks_scenario_table(S0=460, price_impact_pct=price_impact, strikes=[420,440,460,480,500])
for r in rows:
    print(r)

sev_series = np.clip(np.random.default_rng(1).beta(2,6,750), 0, 1)
signal = build_signal_series_from_stress(sev_series)
baseline = simulate_commodity_portfolio(use_signal_overlay=False)
overlay = simulate_commodity_portfolio(use_signal_overlay=True, signal_series=signal, signal_predictive_power=0.35)
print('Baseline annualized alpha:', baseline['alpha_annualized'])
print('Overlay annualized alpha:', overlay['alpha_annualized'])

## Next steps to make this real (not simulated)
1. Get access to a real Pixxel L2A sample cube (or public AVIRIS/EMIT/Hyperion scene) — swap `simulate_dataset()` for `load_real_envi_cube()` in `spectral_physics_simulator.py`. No other code changes needed.
2. Calibrate `stress_score_to_yield_anomaly` against real historical (spectral-index, USDA yield) pairs via regression instead of the illustrative slope used here.
3. Calibrate `yield_anomaly_to_price_impact`'s elasticity against real historical futures-price reactions to USDA WASDE surprises.
4. Replace the synthetic portfolio simulation with real historical commodity futures / options data (e.g. via a data vendor) for a genuine backtest with proper statistical significance testing across many independent periods, not one simulated path.